# Dynamic Programming from Zero

Notebook for **Chapter 9** of the *dsa* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/dsa/ch09.html

Tip: change a number, predict the output, then run the cell again.

## 9.1 What Dynamic Programming Really Is

In [ ]:
calls = 0

def fib(n):
    global calls
    calls += 1                 # count every call we make
    if n < 2:                  # base cases: fib(0)=0, fib(1)=1
        return n
    return fib(n - 1) + fib(n - 2)

for n in (5, 10, 20, 25):
    calls = 0
    value = fib(n)
    print("fib(%d) = %d with %d calls" % (n, value, calls))

fib(5) = 5 with 15 calls
fib(10) = 55 with 177 calls
fib(20) = 6765 with 21891 calls
fib(25) = 75025 with 242785 calls


In [ ]:
calls = 0

def fib_memo(n, memo=None):
    global calls
    calls += 1
    if memo is None:           # fresh dictionary for every top-level call
        memo = {}
    if n in memo:              # card on the wall: reuse it
        return memo[n]
    if n < 2:
        return n
    memo[n] = fib_memo(n - 1, memo) + fib_memo(n - 2, memo)
    return memo[n]

for n in (5, 10, 20, 25, 50):
    calls = 0
    value = fib_memo(n)
    print("fib(%d) = %d with %d calls" % (n, value, calls))

fib(5) = 5 with 9 calls
fib(10) = 55 with 19 calls
fib(20) = 6765 with 39 calls
fib(25) = 75025 with 49 calls
fib(50) = 12586269025 with 99 calls


In [ ]:
def fib_table(n):
    if n < 2:
        return n
    dp = [0] * (n + 1)         # dp[i] holds fib(i)
    dp[1] = 1                  # base cases: dp[0]=0 already
    for i in range(2, n + 1):  # smaller indexes are always ready first
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n]

def fib_o1(n):
    a, b = 0, 1               # a = fib(i), b = fib(i+1)
    for _ in range(n):
        a, b = b, a + b       # slide the two-box window one step
    return a

assert [fib_table(i) for i in range(10)] == [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
assert all(fib_table(i) == fib_o1(i) for i in range(60))
print(fib_table(10), fib_o1(10), fib_o1(90))

55 55 2880067194370816120


## 9.2 The Five-Step Recipe

In [ ]:
from functools import cache

def min_cost_climb(cost):
    n = len(cost)

    @cache                           # the "wall of cards"
    def f(i):                        # STEP 1: state = step i
        if i >= n:                   # STEP 3: base, past the top costs nothing more
            return 0
        # STEP 2: transition, pay for this step, then hop 1 or 2
        return cost[i] + min(f(i + 1), f(i + 2))

    return min(f(0), f(1))           # STEP 5: answer, we may start on 0 or 1

print(min_cost_climb([10, 15, 20]))
print(min_cost_climb([1, 100, 1, 1, 1, 100, 1, 1, 100, 1]))
assert min_cost_climb([10, 15, 20]) == 15
assert min_cost_climb([1, 100, 1, 1, 1, 100, 1, 1, 100, 1]) == 6

15
6


In [ ]:
def min_cost_climb_table(cost):
    n = len(cost)
    f = [0] * (n + 2)                # f[n] = f[n+1] = 0 are the BASE values (step 3)
    for i in range(n - 1, -1, -1):   # STEP 4: order, f[i] needs f[i+1], f[i+2], so go downward
        f[i] = cost[i] + min(f[i + 1], f[i + 2])    # STEP 2: transition
    return min(f[0], f[1])           # STEP 5

tests = [([10, 15, 20], 15), ([1, 100, 1, 1, 1, 100, 1, 1, 100, 1], 6), ([5, 5], 5), ([7], 0)]
for c, expected in tests:
    assert min_cost_climb_table(c) == expected
    print(c, "->", min_cost_climb_table(c))

[10, 15, 20] -> 15
[1, 100, 1, 1, 1, 100, 1, 1, 100, 1] -> 6
[5, 5] -> 5
[7] -> 0


## 9.3 One-Dimensional DP: Stairs, Robbers, Coins and Rising Chains

In [ ]:
def climb(n):
    a, b = 1, 1                   # ways[0], ways[1]
    for _ in range(n - 1):
        a, b = b, a + b
    return b if n >= 1 else 1

def climb_brute(n):
    if n < 0:
        return 0
    if n == 0:
        return 1
    return climb_brute(n - 1) + climb_brute(n - 2)

for n in range(0, 15):
    assert climb(n) == climb_brute(n)
print([climb(n) for n in range(0, 11)])
print(climb(5), climb(45))

[1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]
8 1836311903


In [ ]:
import random
from itertools import product

def rob(money):
    prev2, prev1 = 0, 0            # best[i-2], best[i-1]
    for m in money:
        prev2, prev1 = prev1, max(prev1, prev2 + m)
    return prev1

def rob_brute(money):
    best = 0
    for pick in product([0, 1], repeat=len(money)):
        if any(pick[i] and pick[i + 1] for i in range(len(pick) - 1)):
            continue                # two neighbours chosen, not allowed
        best = max(best, sum(m for m, p in zip(money, pick) if p))
    return best

random.seed(7)
for _ in range(300):
    arr = [random.randint(0, 20) for _ in range(random.randint(0, 9))]
    assert rob(arr) == rob_brute(arr)
print(rob([2, 7, 9, 3, 1]), rob([]), rob([5]), rob([1, 2, 3, 1]))

12 0 5 4


In [ ]:
def coin_change(coins, amount):
    INF = float("inf")
    dp = [0] + [INF] * amount            # dp[0] = 0, the rest unknown
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a and dp[a - c] + 1 < dp[a]:
                dp[a] = dp[a - c] + 1
    return dp[amount] if dp[amount] != INF else -1

def coin_greedy(coins, amount):
    count = 0
    for c in sorted(coins, reverse=True):
        count += amount // c
        amount %= c
    return count if amount == 0 else -1

print(coin_change([1, 3, 4], 6), coin_greedy([1, 3, 4], 6))
print(coin_change([2], 3), coin_change([1, 2, 5], 11), coin_change([5], 0))
assert coin_change([1, 3, 4], 6) == 2
assert coin_change([2], 3) == -1
assert coin_change([1, 2, 5], 11) == 3

2 3
-1 3 0


In [ ]:
import random
from bisect import bisect_left

def lis_n2(a):
    if not a:
        return 0
    dp = [1] * len(a)                     # every element alone is a chain of 1
    for i in range(len(a)):
        for j in range(i):
            if a[j] < a[i]:               # a[i] can extend the chain ending at j
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp)                        # best chain may end anywhere

def lis_fast(a):
    tails = []                            # tails[k] = smallest last value of a chain of length k+1
    for x in a:
        pos = bisect_left(tails, x)       # first tail that is >= x
        if pos == len(tails):
            tails.append(x)               # x extends the longest chain
        else:
            tails[pos] = x                # x is a smaller tail for that length
    return len(tails)

random.seed(3)
for _ in range(200):
    arr = [random.randint(0, 9) for _ in range(random.randint(0, 9))]
    assert lis_n2(arr) == lis_fast(arr)
a = [10, 9, 2, 5, 3, 7, 101, 18]
assert lis_n2(a) == lis_fast(a) == 4 and lis_n2([1, 2, 3, 0]) == 3
print(lis_n2(a), lis_fast(a), lis_n2([]), lis_n2([4, 4, 4]))

4 4 0 1


## 9.4 Two-Dimensional Grid DP

In [ ]:
from math import comb

def unique_paths(rows, cols):
    dp = [[1] * cols for _ in range(rows)]      # border cells start at 1
    for r in range(1, rows):
        for c in range(1, cols):
            dp[r][c] = dp[r - 1][c] + dp[r][c - 1]
    return dp[rows - 1][cols - 1]

for rows in range(1, 8):
    for cols in range(1, 8):
        assert unique_paths(rows, cols) == comb(rows + cols - 2, rows - 1)
print(unique_paths(3, 4), unique_paths(1, 5), unique_paths(7, 3))

10 1 28


In [ ]:
import random
from itertools import combinations

def min_path_sum(grid):
    R, C = len(grid), len(grid[0])
    dp = [row[:] for row in grid]               # copy, so we do not destroy the input
    for r in range(R):
        for c in range(C):
            if r == 0 and c == 0:
                continue
            best_before = min(dp[r - 1][c] if r > 0 else float("inf"),
                              dp[r][c - 1] if c > 0 else float("inf"))
            dp[r][c] += best_before
    return dp[R - 1][C - 1]

def min_path_brute(grid):
    R, C = len(grid), len(grid[0])
    best = float("inf")
    for downs in combinations(range(R + C - 2), R - 1):   # which of the moves are "down"
        r = c = 0
        total = grid[0][0]
        for move in range(R + C - 2):
            if move in downs:
                r += 1
            else:
                c += 1
            total += grid[r][c]
        best = min(best, total)
    return best

random.seed(5)
for _ in range(100):
    R, C = random.randint(1, 4), random.randint(1, 4)
    g = [[random.randint(0, 9) for _ in range(C)] for _ in range(R)]
    assert min_path_sum(g) == min_path_brute(g)
print(min_path_sum([[1, 3, 1], [1, 5, 1], [4, 2, 1]]), min_path_sum([[7]]), min_path_sum([[1, 2, 3]]))

7 7 6


## 9.5 Knapsack: 0/1, Unbounded and Subset Sum

In [ ]:
import random
from itertools import product

def knapsack_2d(items, W):
    n = len(items)
    dp = [[0] * (W + 1) for _ in range(n + 1)]          # row 0 is the base
    for i in range(1, n + 1):
        w, v = items[i - 1]
        for c in range(W + 1):
            dp[i][c] = dp[i - 1][c]                      # leave it
            if c >= w:
                dp[i][c] = max(dp[i][c], dp[i - 1][c - w] + v)   # take it
    return dp[n][W]

def knapsack_1d(items, W):
    dp = [0] * (W + 1)
    for w, v in items:
        for c in range(W, w - 1, -1):                    # BACKWARD so each item is used once
            dp[c] = max(dp[c], dp[c - w] + v)
    return dp[W]

def knapsack_brute(items, W):
    best = 0
    for pick in product([0, 1], repeat=len(items)):
        weight = sum(w for (w, v), p in zip(items, pick) if p)
        if weight <= W:
            best = max(best, sum(v for (w, v), p in zip(items, pick) if p))
    return best

items = [(2, 3), (3, 4), (4, 5), (5, 6)]
random.seed(11)
for _ in range(200):
    its = [(random.randint(1, 6), random.randint(1, 9)) for _ in range(random.randint(0, 7))]
    cap = random.randint(0, 12)
    assert knapsack_2d(its, cap) == knapsack_1d(its, cap) == knapsack_brute(its, cap)
print(knapsack_2d(items, 5), knapsack_1d(items, 5))

7 7


In [ ]:
def unbounded_knapsack(items, W):
    dp = [0] * (W + 1)
    for w, v in items:
        for c in range(w, W + 1):                # FORWARD: the item can be reused
            dp[c] = max(dp[c], dp[c - w] + v)
    return dp[W]

items = [(2, 3), (3, 4), (4, 5), (5, 6)]
print("unbounded:", unbounded_knapsack(items, 5))
assert unbounded_knapsack([(2, 3), (3, 4)], 6) == 9
assert unbounded_knapsack([(5, 10)], 4) == 0

unbounded: 7


In [ ]:
def ways_combinations(coins, amount):
    dp = [1] + [0] * amount                      # one way to make 0: take nothing
    for c in coins:                              # coins OUTSIDE: each coin is "decided" once
        for a in range(c, amount + 1):
            dp[a] += dp[a - c]
    return dp[amount]

print(ways_combinations([1, 2, 5], 5))
assert ways_combinations([1, 2, 5], 5) == 4      # 5, 2+2+1, 2+1+1+1, 1+1+1+1+1
assert ways_combinations([2], 3) == 0
assert ways_combinations([7], 0) == 1

4


In [ ]:
def subset_sum(nums, target):
    can = [False] * (target + 1)
    can[0] = True                                # the empty subset sums to 0
    for x in nums:
        for s in range(target, x - 1, -1):       # backward: use each number once
            if can[s - x]:
                can[s] = True
    return can[target]

def can_partition(nums):
    total = sum(nums)
    return total % 2 == 0 and subset_sum(nums, total // 2)

print(subset_sum([3, 4, 5], 9), subset_sum([3, 4, 5], 6), subset_sum([], 0))
print(can_partition([1, 5, 11, 5]), can_partition([1, 2, 3, 5]))
assert subset_sum([3, 4, 5], 9) and not subset_sum([3, 4, 5], 6)
assert can_partition([1, 5, 11, 5]) and not can_partition([1, 2, 3, 5])

True False True
True False


## 9.6 String DP: LCS, Edit Distance and Palindromes

In [ ]:
import random

def lcs(a, b):
    n, m = len(a), len(b)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1           # letters match: extend the diagonal
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])  # drop one letter from either side
    # rebuild one actual LCS by walking back from the corner
    out = []
    i, j = n, m
    while i > 0 and j > 0:
        if a[i - 1] == b[j - 1]:
            out.append(a[i - 1])
            i -= 1
            j -= 1
        elif dp[i - 1][j] >= dp[i][j - 1]:
            i -= 1
        else:
            j -= 1
    return dp[n][m], "".join(reversed(out))

def is_subseq(s, t):
    it = iter(t)
    return all(ch in it for ch in s)

random.seed(2)
for _ in range(150):
    x = "".join(random.choice("abc") for _ in range(random.randint(0, 7)))
    y = "".join(random.choice("abc") for _ in range(random.randint(0, 7)))
    length, text = lcs(x, y)
    assert length == len(text)
    assert is_subseq(text, x) and is_subseq(text, y)
assert lcs("AGGTAB", "GXTXAYB") == (4, "GTAB") and lcs("ABCBDAB", "BDCABA")[0] == 4
print(lcs("AGGTAB", "GXTXAYB"))
print(lcs("abc", ""), lcs("abc", "abc"), lcs("abc", "def"))

(4, 'GTAB')
(0, '') (3, 'abc') (0, '')


In [ ]:
import random

def edit_distance(a, b):
    n, m = len(a), len(b)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = i                       # delete all i letters
    for j in range(m + 1):
        dp[0][j] = j                       # insert all j letters
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j - 1], dp[i - 1][j], dp[i][j - 1])
    return dp[n][m]

random.seed(4)
for _ in range(200):
    x = "".join(random.choice("abc") for _ in range(random.randint(0, 6)))
    y = "".join(random.choice("abc") for _ in range(random.randint(0, 6)))
    assert edit_distance(x, y) == edit_distance(y, x)       # the distance is symmetric
assert edit_distance("kitten", "sitting") == 3
print(edit_distance("horse", "ros"), edit_distance("intention", "execution"), edit_distance("", "abc"), edit_distance("same", "same"))

3 5 3 0


In [ ]:
def count_pal_expand(s):
    count = 0
    for centre in range(2 * len(s) - 1):
        left = centre // 2
        right = left + centre % 2                  # odd centre: left == right; even: neighbours
        while left >= 0 and right < len(s) and s[left] == s[right]:
            count += 1
            left -= 1
            right += 1
    return count

for s in ["", "a", "abc", "aaa", "abba", "racecar", "abacdfgdcaba"]:
    assert count_pal_expand(s) == sum(s[i:j] == s[i:j][::-1] for i in range(len(s)) for j in range(i + 1, len(s) + 1))
print(count_pal_expand("aaa"), count_pal_expand("abc"), count_pal_expand("racecar"))

6 3 10


## 9.7 Interval DP and Bitmask Awareness

In [ ]:
def lps(s):
    n = len(s)
    if n == 0:
        return 0
    dp = [[0] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):               # i from the end so row i+1 is ready
        dp[i][i] = 1
        for j in range(i + 1, n):
            if s[i] == s[j]:
                inside = dp[i + 1][j - 1] if i + 1 <= j - 1 else 0
                dp[i][j] = inside + 2
            else:
                dp[i][j] = max(dp[i + 1][j], dp[i][j - 1])
    return dp[0][n - 1]

for s, expected in [("", 0), ("a", 1), ("bbbab", 4), ("cbbd", 2), ("AXBYBA", 5), ("abcde", 1)]:
    assert lps(s) == expected
print(lps("AXBYBA"), lps("bbbab"), lps("abcde"))

5 4 1


In [ ]:
from itertools import permutations

def burst(nums):
    a = [1] + nums + [1]
    n = len(a)
    dp = [[0] * n for _ in range(n)]
    for length in range(2, n):                   # distance between the two fixed ends
        for l in range(n - length):
            r = l + length
            for k in range(l + 1, r):            # k is the LAST balloon burst inside (l, r)
                coins = dp[l][k] + dp[k][r] + a[l] * a[k] * a[r]
                dp[l][r] = max(dp[l][r], coins)
    return dp[0][n - 1]

def burst_brute(nums):
    best = 0
    for order in permutations(range(len(nums))):
        alive = list(range(len(nums)))
        total = 0
        for k in order:
            pos = alive.index(k)
            left = nums[alive[pos - 1]] if pos > 0 else 1
            right = nums[alive[pos + 1]] if pos + 1 < len(alive) else 1
            total += left * nums[k] * right
            alive.pop(pos)
        best = max(best, total)
    return best

for arr in [[], [4], [3, 1, 5], [3, 1, 5, 8]]:
    assert burst(arr) == burst_brute(arr)
print(burst([3, 1, 5]), burst([3, 1, 5, 8]), burst([]))

35 167 0


In [ ]:
from itertools import permutations

def assign(cost):
    n = len(cost)
    INF = float("inf")
    dp = [INF] * (1 << n)
    dp[0] = 0                                    # no tasks done, no cost
    for mask in range(1 << n):
        if dp[mask] == INF:
            continue
        w = bin(mask).count("1")                 # next worker to place
        if w == n:
            continue
        for t in range(n):
            if not mask & (1 << t):              # task t still free
                nxt = mask | (1 << t)
                dp[nxt] = min(dp[nxt], dp[mask] + cost[w][t])
    return dp[(1 << n) - 1]

def assign_brute(cost):
    n = len(cost)
    return min(sum(cost[w][p[w]] for w in range(n)) for p in permutations(range(n)))

cost = [[9, 2, 7], [6, 4, 3], [5, 8, 1]]
assert assign(cost) == assign_brute(cost)
assert assign([[4]]) == 4
print(assign(cost))

9


## 9.8 Recognising DP, Saving Space and Rebuilding the Answer

In [ ]:
def lcs_two_rows(a, b):
    prev = [0] * (len(b) + 1)                    # row i-1
    for i in range(1, len(a) + 1):
        cur = [0] * (len(b) + 1)                 # row i
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                cur[j] = prev[j - 1] + 1
            else:
                cur[j] = max(prev[j], cur[j - 1])
        prev = cur                               # the old row is thrown away
    return prev[-1]

def unique_paths_one_row(rows, cols):
    row = [1] * cols                             # row 0
    for _ in range(1, rows):
        for c in range(1, cols):
            row[c] += row[c - 1]                 # row[c] is "above", row[c-1] is "left"
    return row[-1]

print(lcs_two_rows("AGGTAB", "GXTXAYB"), unique_paths_one_row(3, 4), unique_paths_one_row(10, 10))
assert lcs_two_rows("AGGTAB", "GXTXAYB") == 4
assert unique_paths_one_row(10, 10) == 48620

4 10 48620


In [ ]:
def coin_change_coins(coins, amount):
    INF = float("inf")
    dp = [0] + [INF] * amount
    last_coin = [0] * (amount + 1)               # parent pointer: the coin that gave dp[a]
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a and dp[a - c] + 1 < dp[a]:
                dp[a] = dp[a - c] + 1
                last_coin[a] = c
    if dp[amount] == INF:
        return None
    used, a = [], amount
    while a > 0:
        used.append(last_coin[a])                # follow the pointers back to 0
        a -= last_coin[a]
    return used

print(coin_change_coins([1, 3, 4], 6), coin_change_coins([2], 3), coin_change_coins([1, 2, 5], 11))
assert sum(coin_change_coins([1, 2, 5], 11)) == 11

[3, 3] None [1, 5, 5]


## Chapter recap

## Practice set

Graded from easy to hard. Try each one on paper first, then read the answer.

In [ ]:
dp = [0] * 6
dp[1] = 1
for i in range(2, 6):
    dp[i] = dp[i - 1] + dp[i - 2]
print(dp)

In [ ]:
def knapsack_bug(items, W):
    dp = [0] * (W + 1)
    for w, v in items:
        for c in range(w, W + 1):        # loop direction?
            dp[c] = max(dp[c], dp[c - w] + v)
    return dp[W]

print(knapsack_bug([(2, 3)], 5))

6


In [ ]:
def lis_bug(a):
    dp = [1] * len(a)
    for i in range(len(a)):
        for j in range(i):
            if a[j] < a[i]:
                dp[i] = max(dp[i], dp[j] + 1)
    return dp[-1]

print(lis_bug([1, 2, 3, 0]))

1


In [ ]:
def combos(coins, amount):
    dp = [1] + [0] * amount
    for c in coins:
        for a in range(c, amount + 1):
            dp[a] += dp[a - c]
    return dp[amount]

def orders(coins, amount):
    dp = [1] + [0] * amount
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a:
                dp[a] += dp[a - c]
    return dp[amount]

print(combos([1, 2], 3), orders([1, 2], 3))

In [ ]:
def edit_bug(a, b):
    n, m = len(a), len(b)
    dp = [[0] * (m + 1) for _ in range(n + 1)]     # borders left at 0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j - 1], dp[i - 1][j], dp[i][j - 1])
    return dp[n][m]

print(edit_bug("abc", ""), edit_bug("horse", "ros"))

0 2


In [ ]:
dp = [[0] * 2] * 3
dp[1][1] = 9
print(dp)

In [ ]:
def decode_ways(s):
    if not s or s[0] == "0":
        return 0
    prev2, prev1 = 1, 1                     # ways for 0 digits and for 1 digit
    for i in range(2, len(s) + 1):
        cur = 0
        if s[i - 1] != "0":
            cur += prev1                    # last digit alone
        if 10 <= int(s[i - 2:i]) <= 26:
            cur += prev2                    # last two digits together
        prev2, prev1 = prev1, cur
    return prev1

print(decode_ways("226"), decode_ways("06"), decode_ways("10"), decode_ways("27"))
assert decode_ways("226") == 3 and decode_ways("12") == 2

3 0 1 1


## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/dsa/ch09.html